# Advanced Demo — Equal-Weight M7 Portfolio, 2015
## MATH GR 5320

This notebook builds an equal-weight **Magnificent Seven** (M7) portfolio priced at 2015-12-31,
adds European option positions, and runs the full risk suite against it.

| § | Topic |
|---|---|
| 1 | M7 portfolio construction and data download |
| 2 | Stock-only baseline: Historical · Parametric · Monte Carlo VaR/ES |
| 3 | Diversification benefit across M7 |
| 4 | Add option positions (OTM calls on AAPL & AMZN; OTM short puts on TSLA) |
| 5 | Full portfolio risk (stocks + options) |
| 6 | VaR backtesting on 2013–2016 data |
| 7 | Credit extensions: Merton Q-PD and P-PD for NVDA and TSLA |
| 8 | Frontend validation summary — numbers replicated in the Streamlit app |

All calculations call `src/` modules directly via `RiskEngineService`.
The companion `advanced_demo.md` captures the Streamlit front-end trace.

> **yfinance note**: prices are split-adjusted to today, so 2015 AAPL ≈ \$26 (post 4:1 2020),
> GOOGL ≈ \$38 (post 20:1 2022), NVDA ≈ \$0.80 (post 4:1 2021 and 10:1 2024), etc.
> The return series and risk numbers are economically correct regardless of scale.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
from datetime import date
from IPython.display import display

from src.data.market_data import download_adjusted_close
from src.schemas import Portfolio, StockPosition, OptionPosition
from src.portfolio.portfolio import portfolio_value, portfolio_exposure
from src.risk.historical import historical_var_es
from src.risk.parametric import parametric_var_es
from src.risk.monte_carlo import monte_carlo_var_es
from src.risk.backtest import run_backtest, kupiec_test
from src.credit.merton import merton_pd, merton_equity, merton_debt
from src.pricing.black_scholes import bs_price, bs_delta
from src.services.risk_engine_service import RiskEngineService

pd.set_option('display.float_format', '{:,.4f}'.format)
print('Imports OK')

Imports OK


---
## §1 — M7 Portfolio Construction

Equal notional of **\$50 000 per stock** (\$350 000 total). Share counts are computed
from the yfinance split-adjusted close on 2015-12-31.

In [2]:
# ── Tickers and date range ──────────────────────────────────────────────
M7         = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA']
EVAL_DATE  = date(2015, 12, 31)
DATA_START = '2013-01-01'
DATA_END   = '2016-12-31'

prices = download_adjusted_close(M7, DATA_START, DATA_END)
prices = prices.dropna()

# History available up to and including the evaluation date
prices_to_eval = prices[prices.index <= pd.Timestamp(EVAL_DATE)]
spot_prices    = prices_to_eval.iloc[-1]

print(f'Full history   : {len(prices)} days  {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Up to eval     : {len(prices_to_eval)} days')
print(f'\nSplit-adjusted close prices on {EVAL_DATE}:')
spot_df = spot_prices.rename('Adj. Close ($)').to_frame()
display(spot_df.round(4))

Full history   : 1008 days  2013-01-02 → 2016-12-30
Up to eval     : 756 days

Split-adjusted close prices on 2015-12-31:


,Adj. Close ($)
Ticker,
AAPL,23.7107
AMZN,33.7945
GOOGL,38.5816
META,103.8451
MSFT,48.4682
NVDA,0.8039
TSLA,16.0007


In [3]:
# ── Equal-weight portfolio — $50k per stock ────────────────────────────
NOTIONAL = 50_000.0

shares = {t: int(NOTIONAL / spot_prices[t]) for t in M7}

stock_portfolio = Portfolio(
    stocks=[StockPosition(ticker=t, quantity=shares[t]) for t in M7]
)

V0_stocks = portfolio_value(stock_portfolio, spot_prices, EVAL_DATE)

comp = pd.DataFrame({
    'Shares'     : pd.Series(shares),
    'Price ($)'  : spot_prices.round(4),
    'Notional ($)': {t: round(shares[t] * spot_prices[t], 2) for t in M7},
    'Weight (%)'  : {t: round(shares[t] * spot_prices[t] / V0_stocks * 100, 2) for t in M7},
})
comp.index.name = 'Ticker'

print(f'Portfolio value (stocks only): ${V0_stocks:,.2f}')
print()
display(comp)

# Save CSV for the Streamlit frontend
csv_path = os.path.join('..', 'data', 'm7_2015.csv')
prices.to_csv(csv_path)
print(f'\nSaved M7 price history → {csv_path}')
print('→ Upload this file in Tab 2 (Market Data) of the Streamlit app.')

Portfolio value (stocks only): $349,833.69



,Shares,Price ($),Notional ($),Weight (%)
Ticker,,,,
AAPL,2108,23.7107,"49,982.1000",14.2900
AMZN,1479,33.7945,"49,982.0600",14.2900
GOOGL,1295,38.5816,"49,963.2100",14.2800
META,481,103.8451,"49,949.4900",14.2800
MSFT,1031,48.4682,"49,970.7600",14.2800
NVDA,62194,0.8039,"49,999.9900",14.2900
TSLA,3124,16.0007,"49,986.0900",14.2900



Saved M7 price history → ../data/m7_2015.csv
→ Upload this file in Tab 2 (Market Data) of the Streamlit app.


---
## §2 — Stock-Only Baseline: Historical · Parametric · Monte Carlo

**Risk parameters** used throughout:

| Parameter | Value |
|---|---|
| Lookback | 252 trading days |
| Horizon | 5 trading days |
| VaR confidence | 99% |
| ES confidence | 97.5% |
| Estimator | rolling window |
| MC paths | 10 000 (seed 42) |

In [4]:
# ── Risk parameters ──────────────────────────────────────────────────────
LOOKBACK  = 252
HORIZON   = 5
VAR_CONF  = 0.99
ES_CONF   = 0.975
N_SIMS    = 10_000
SEED      = 42

svc_stocks = RiskEngineService(
    portfolio      = stock_portfolio,
    prices         = prices_to_eval,
    pricing_date   = EVAL_DATE,
    lookback_days  = LOOKBACK,
    horizon_days   = HORIZON,
    var_confidence = VAR_CONF,
    es_confidence  = ES_CONF,
    n_simulations  = N_SIMS,
    estimator      = 'window',
)

print(f'Portfolio value: ${svc_stocks.portfolio_value():,.2f}')
print(f'Running all three risk methods … ', end='')
results_stocks = svc_stocks.run_all()
print('done.')

Portfolio value: $349,833.69
Running all three risk methods … 

done.


In [5]:
r_hist  = results_stocks['historical']
r_param = results_stocks['parametric']
r_mc    = results_stocks['monte_carlo']

comparison = pd.DataFrame({
    'VaR 99% 5d ($)' : [r_hist['var'],  r_param['var'],  r_mc['var']],
    'ES 97.5% 5d ($)': [r_hist['es'],   r_param['es'],   r_mc['es']],
    'ES / VaR'       : [r_hist['es']  / r_hist['var'],
                        r_param['es'] / r_param['var'],
                        r_mc['es']    / r_mc['var']],
    'VaR / V0 (%)'   : [r_hist['var']  / V0_stocks * 100,
                        r_param['var'] / V0_stocks * 100,
                        r_mc['var']    / V0_stocks * 100],
}, index=['Historical', 'Parametric', 'Monte Carlo'])
comparison.index.name = 'Method'

print(f'Stock-only M7  |  VaR {VAR_CONF:.0%} / ES {ES_CONF:.1%}  |  {HORIZON}-day horizon')
display(comparison.round(4))

# Structural assertions
for name, r in [('Historical', r_hist), ('Parametric', r_param), ('Monte Carlo', r_mc)]:
    assert r['var'] > 0,          f'{name}: VaR must be positive'
    assert r['es'] >= r['var'],   f'{name}: ES must be >= VaR'

print()
print('✓ VaR > 0 for all three methods')
print('✓ ES >= VaR for all three methods')

Stock-only M7  |  VaR 99% / ES 97.5%  |  5-day horizon


,VaR 99% 5d ($),ES 97.5% 5d ($),ES / VaR,VaR / V0 (%)
Method,,,,
Historical,"25,470.5776","27,483.8249",1.0790,7.2808
Parametric,"22,161.6850","22,281.6402",1.0054,6.3349
Monte Carlo,"21,683.5250","21,886.1296",1.0093,6.1982



✓ VaR > 0 for all three methods
✓ ES >= VaR for all three methods


---
## §3 — Diversification Benefit

A diversified portfolio should satisfy **sub-additivity**: its VaR should be strictly less
than the sum of the seven individual stock VaRs.

In [6]:
stock_vars = {}
for t in M7:
    single_port = Portfolio(stocks=[StockPosition(ticker=t, quantity=shares[t])])
    r_single = historical_var_es(
        portfolio      = single_port,
        prices         = prices_to_eval[[t]],
        pricing_date   = EVAL_DATE,
        lookback_days  = LOOKBACK,
        horizon_days   = HORIZON,
        var_confidence = VAR_CONF,
        es_confidence  = ES_CONF,
    )
    stock_vars[t] = r_single['var']

sum_indiv_var  = sum(stock_vars.values())
port_var_hist  = r_hist['var']
diversif_ratio = 1.0 - port_var_hist / sum_indiv_var

diversif_df = pd.DataFrame({
    'Individual VaR ($)': stock_vars,
    'Notional ($)'      : {t: round(shares[t] * spot_prices[t], 2) for t in M7},
    'VaR / Notional (%)': {t: stock_vars[t] / (shares[t] * spot_prices[t]) * 100 for t in M7},
})
diversif_df.index.name = 'Ticker'

display(diversif_df.round(2))
print(f'\nSum of individual VaRs  : ${sum_indiv_var:,.2f}')
print(f'Portfolio VaR (hist)    : ${port_var_hist:,.2f}')
print(f'Diversification benefit : {diversif_ratio:.1%}')

assert port_var_hist < sum_indiv_var, 'Portfolio VaR < sum of individual VaRs (sub-additivity)'
print('\n✓ Portfolio VaR < sum of individual VaRs — diversification confirmed')

,Individual VaR ($),Notional ($),VaR / Notional (%)
Ticker,,,
AAPL,"3,990.4000","49,982.1000",7.9800
MSFT,"5,874.0400","49,970.7600",11.7500
GOOGL,"3,243.6800","49,963.2100",6.4900
AMZN,"3,983.9300","49,982.0600",7.9700
NVDA,"4,644.8100","49,999.9900",9.2900
META,"4,337.3400","49,949.4900",8.6800
TSLA,"6,079.0500","49,986.0900",12.1600



Sum of individual VaRs  : $32,153.24
Portfolio VaR (hist)    : $25,470.58
Diversification benefit : 20.8%

✓ Portfolio VaR < sum of individual VaRs — diversification confirmed


---
## §4 — Option Positions

Three option positions are added to test the impact on the risk profile:

| # | Position | Rationale |
|---|---|---|
| 1 | +10 AAPL calls (5% OTM, Jun-2016) | Long upside exposure; bounded left-tail loss |
| 2 | +5 AMZN calls (8% OTM, Jun-2016) | Long growth; bounded downside |
| 3 | −8 TSLA puts (10% OTM, Mar-2016) | Short puts = additional downside risk below the strike |

Strikes are set as percentages of the split-adjusted 2015-12-31 spot price.

In [7]:
S_aapl = float(spot_prices['AAPL'])
S_amzn = float(spot_prices['AMZN'])
S_tsla = float(spot_prices['TSLA'])

K_aapl = round(S_aapl * 1.05, 4)   # 5% OTM call
K_amzn = round(S_amzn * 1.08, 4)   # 8% OTM call
K_tsla = round(S_tsla * 0.90, 4)   # 10% OTM put (short)

R_FREE = 0.02   # simplified 2% risk-free (Dec-2015 ≈ 0.5–1%; 2% is conservative)

options = [
    OptionPosition(
        ticker              = 'AAPL_C_OTM',
        underlying_ticker   = 'AAPL',
        option_type         = 'call',
        quantity            = 10,
        strike              = K_aapl,
        maturity_date       = date(2016, 6, 30),
        volatility          = 0.25,
        risk_free_rate      = R_FREE,
        dividend_yield      = 0.02,
        contract_multiplier = 100.0,
    ),
    OptionPosition(
        ticker              = 'AMZN_C_OTM',
        underlying_ticker   = 'AMZN',
        option_type         = 'call',
        quantity            = 5,
        strike              = K_amzn,
        maturity_date       = date(2016, 6, 30),
        volatility          = 0.30,
        risk_free_rate      = R_FREE,
        dividend_yield      = 0.0,
        contract_multiplier = 100.0,
    ),
    OptionPosition(
        ticker              = 'TSLA_P_OTM',
        underlying_ticker   = 'TSLA',
        option_type         = 'put',
        quantity            = -8,
        strike              = K_tsla,
        maturity_date       = date(2016, 3, 31),
        volatility          = 0.55,
        risk_free_rate      = R_FREE,
        dividend_yield      = 0.0,
        contract_multiplier = 100.0,
    ),
]

print(f'Option strikes at {EVAL_DATE}:')
print(f'  AAPL call  K = {K_aapl:.4f}  (105% of {S_aapl:.4f})')
print(f'  AMZN call  K = {K_amzn:.4f}  (108% of {S_amzn:.4f})')
print(f'  TSLA put   K = {K_tsla:.4f}  ( 90% of {S_tsla:.4f})')

Option strikes at 2015-12-31:
  AAPL call  K = 24.8962  (105% of 23.7107)
  AMZN call  K = 36.4981  (108% of 33.7945)
  TSLA put   K = 14.4006  ( 90% of 16.0007)


In [8]:
T_jun = (date(2016, 6, 30) - EVAL_DATE).days / 365.0
T_mar = (date(2016, 3, 31) - EVAL_DATE).days / 365.0

opt_rows = []
for opt in options:
    T  = T_jun if opt.maturity_date == date(2016, 6, 30) else T_mar
    S  = float(spot_prices[opt.underlying_ticker])
    px = bs_price(S, opt.strike, T, opt.risk_free_rate, opt.dividend_yield, opt.volatility, opt.option_type)
    dl = bs_delta(S, opt.strike, T, opt.risk_free_rate, opt.dividend_yield, opt.volatility, opt.option_type)
    opt_rows.append({
        'Ticker'           : opt.ticker,
        'Type'             : opt.option_type,
        'Qty'              : int(opt.quantity),
        'K'                : round(opt.strike, 4),
        'S'                : round(S, 4),
        'T (yr)'           : round(T, 3),
        'σ'                : opt.volatility,
        'BS Price'         : round(px, 6),
        'Delta'            : round(dl, 6),
        'Total Value ($)'  : round(opt.quantity * opt.contract_multiplier * px, 4),
        'Dollar Δ ($)'     : round(opt.quantity * opt.contract_multiplier * dl * S, 4),
    })

opt_df = pd.DataFrame(opt_rows).set_index('Ticker')
print('Option positions (Black-Scholes at evaluation date):')
display(opt_df)

# Build the full portfolio
full_portfolio = Portfolio(stocks=stock_portfolio.stocks, options=options)
V0_full = portfolio_value(full_portfolio, spot_prices, EVAL_DATE)

print(f'\nPortfolio value — stocks only  : ${V0_stocks:,.4f}')
print(f'Portfolio value — full (+ opts): ${V0_full:,.4f}')
print(f'Options net value              : ${V0_full - V0_stocks:,.4f}')

Option positions (Black-Scholes at evaluation date):


,Type,Qty,K,S,T (yr),σ,BS Price,Delta,Total Value ($),Dollar Δ ($)
Ticker,,,,,,,,,,
AAPL_C_OTM,call,10,24.8962,23.7107,0.4990,0.2500,1.1696,0.4212,"1,169.6011","9,986.3264"
AMZN_C_OTM,call,5,36.4981,33.7945,0.4990,0.3000,1.9244,0.4167,962.2025,"7,041.3356"
TSLA_P_OTM,put,-8,14.4006,16.0007,0.2490,0.5500,0.9513,-0.2949,-761.0590,"3,774.8872"



Portfolio value — stocks only  : $349,833.6928
Portfolio value — full (+ opts): $351,204.4375
Options net value              : $1,370.7446


---
## §5 — Full Portfolio Risk: Stocks + Options

Same risk parameters as §2. We compare VaR/ES with and without the option positions.

In [9]:
svc_full = RiskEngineService(
    portfolio      = full_portfolio,
    prices         = prices_to_eval,
    pricing_date   = EVAL_DATE,
    lookback_days  = LOOKBACK,
    horizon_days   = HORIZON,
    var_confidence = VAR_CONF,
    es_confidence  = ES_CONF,
    n_simulations  = N_SIMS,
    estimator      = 'window',
)

print('Running all three methods on full portfolio (stocks + options) … ', end='')
results_full = svc_full.run_all()
print('done.')

rf_hist  = results_full['historical']
rf_param = results_full['parametric']
rf_mc    = results_full['monte_carlo']

Running all three methods on full portfolio (stocks + options) … 

done.


In [10]:
impact = pd.DataFrame({
    'VaR Stocks ($)': [r_hist['var'],  r_param['var'],  r_mc['var']],
    'VaR Full ($)'  : [rf_hist['var'], rf_param['var'], rf_mc['var']],
    'VaR Δ ($)'     : [rf_hist['var']  - r_hist['var'],
                       rf_param['var'] - r_param['var'],
                       rf_mc['var']    - r_mc['var']],
    'ES Stocks ($)' : [r_hist['es'],   r_param['es'],   r_mc['es']],
    'ES Full ($)'   : [rf_hist['es'],  rf_param['es'],  rf_mc['es']],
    'ES Δ ($)'      : [rf_hist['es']  - r_hist['es'],
                       rf_param['es'] - r_param['es'],
                       rf_mc['es']    - r_mc['es']],
}, index=['Historical', 'Parametric', 'Monte Carlo'])
impact.index.name = 'Method'

print('Impact of options on VaR and ES:')
display(impact.round(4))

# Structural assertions on full portfolio
for name, rf in [('Historical', rf_hist), ('Parametric', rf_param), ('Monte Carlo', rf_mc)]:
    assert rf['var'] > 0,          f'{name}: Full portfolio VaR must be > 0'
    assert rf['es'] >= rf['var'],  f'{name}: Full portfolio ES >= VaR'

print()
print('✓ Full portfolio VaR > 0 for all methods')
print('✓ Full portfolio ES >= VaR for all methods')

hist_impact = rf_hist['var'] - r_hist['var']
print(f'\nHistorical VaR change from adding options: ${hist_impact:+,.4f}')
if hist_impact > 0:
    print('  → Net effect: options increase downside risk (short TSLA puts dominate)')
else:
    print('  → Net effect: options reduce downside risk (long call delta offsets short put risk)')

Impact of options on VaR and ES:


,VaR Stocks ($),VaR Full ($),VaR Δ ($),ES Stocks ($),ES Full ($),ES Δ ($)
Method,,,,,,
Historical,"25,470.5776","26,802.5954","1,332.0177","27,483.8249","29,004.3964","1,520.5716"
Parametric,"22,161.6850","23,519.2367","1,357.5517","22,281.6402","23,646.4102","1,364.7699"
Monte Carlo,"21,683.5250","22,747.8099","1,064.2849","21,886.1296","23,098.0683","1,211.9388"



✓ Full portfolio VaR > 0 for all methods
✓ Full portfolio ES >= VaR for all methods

Historical VaR change from adding options: $+1,332.0177
  → Net effect: options increase downside risk (short TSLA puts dominate)


---
## §6 — VaR Backtesting

Walk-forward historical backtest over the full 2013–2016 period.
- Estimation window: 252 trading days.
- Forecast horizon: 5 trading days.
- Each forecast step uses only data available at that point (out-of-sample).

The Kupiec LR test checks unconditional coverage: are exceptions clustering at the expected rate?

In [11]:
# Use stock-only portfolio for a clean, fast backtest
svc_bt = RiskEngineService(
    portfolio      = stock_portfolio,
    prices         = prices,           # full 2013-2016 range
    pricing_date   = EVAL_DATE,
    lookback_days  = LOOKBACK,
    horizon_days   = HORIZON,
    var_confidence = VAR_CONF,
    es_confidence  = ES_CONF,
    n_simulations  = 2_000,           # capped for speed
    estimator      = 'window',
)

print('Running walk-forward backtest … ', end='')
bt_result = svc_bt.run_backtest(model='historical')
print('done.')

bt_df  = bt_result['backtest_df']
kupiec = bt_result['kupiec']

alpha_bt = 1.0 - VAR_CONF
exp_exc  = kupiec['n_observations'] * alpha_bt

print(f'\nBacktest observations  : {kupiec["n_observations"]}')
print(f'Expected exceptions    : {exp_exc:.1f}  ({alpha_bt:.1%} of obs)')
print(f'Actual exceptions      : {kupiec["n_exceptions"]}')
print(f'Exception rate (p_hat) : {kupiec["p_hat"]:.4%}')
print(f'Kupiec LR statistic    : {kupiec["lr_stat"]:.4f}')
print(f'Kupiec p-value         : {kupiec["p_value"]:.4f}')
print(f'Reject H0 (5% level)   : {"Yes ✗" if kupiec["reject_h0"] else "No ✓"}')

Running walk-forward backtest … 

done.

Backtest observations  : 750
Expected exceptions    : 7.5  (1.0% of obs)
Actual exceptions      : 18
Exception rate (p_hat) : 2.4000%
Kupiec LR statistic    : 10.6661
Kupiec p-value         : 0.0011
Reject H0 (5% level)   : Yes ✗


In [12]:
# Kupiec assertion
if not kupiec['reject_h0']:
    print('✓ Kupiec: historical VaR model not rejected at 5% significance on 2013-2016 M7 data')
else:
    print('⚠ Kupiec: model rejected — examine clustering or window sensitivity')

# Show first few exception dates
if not bt_df.empty and int(bt_df['exception'].sum()) > 0:
    exc_df = bt_df[bt_df['exception'] == 1][['date', 'var_forecast', 'realized_loss']].head(10)
    exc_df = exc_df.rename(columns={
        'date': 'Date', 'var_forecast': 'VaR Forecast ($)', 'realized_loss': 'Realised Loss ($)'
    })
    print(f'\nException dates (first 10 of {int(bt_df["exception"].sum())}):')
    display(exc_df.round(2))
else:
    print('\n(No exceptions or empty backtest DataFrame.)')

⚠ Kupiec: model rejected — examine clustering or window sensitivity

Exception dates (first 10 of 18):


,Date,VaR Forecast ($),Realised Loss ($)
12,2014-01-22,"9,341.2100","9,493.0000"
51,2014-03-19,"10,340.5200","12,652.7300"
52,2014-03-20,"10,333.4400","14,936.0200"
53,2014-03-21,"10,197.7700","11,123.6800"
81,2014-05-01,"10,172.9100","11,216.4800"
190,2014-10-06,"13,162.9600","17,627.0100"
192,2014-10-08,"13,094.1400","18,052.6400"
193,2014-10-09,"12,912.5500","16,658.5000"
235,2014-12-09,"15,169.4800","16,357.5800"
406,2015-08-14,"18,783.2500","23,262.2800"


---
## §7 — Credit Extensions: Merton Structural Default Model

Apply the Merton (1974) model to **NVDA** and **TSLA** using end-2015 estimates:

- **V₀** ≈ market cap + book long-term debt (simplified)
- **B** = face value of long-term debt
- **σ_A** = asset vol ≈ equity vol × (equity / V₀)  *(Merton leverage adjustment)*
- **Q-PD** = N(−d₂) with ν = r (risk-neutral drift)
- **P-PD** = N(−d₂) with ν = μ̂ (historical annual drift from trailing returns)

End-2015 inputs (in \$B, sourced from public filings):

| Ticker | Mkt cap (\$B) | LT debt (\$B) | σ_E (hist.) |
|---|---|---|---|
| NVDA | ~15 | ~1.3 | computed from data |
| TSLA | ~31 | ~2.7 | computed from data |

In [13]:
# ── Estimate trailing 252-day equity vol and annual drift ─────────────────
log_ret = (np.log(prices_to_eval / prices_to_eval.shift(1))).dropna()

for t in ['NVDA', 'TSLA']:
    sigma_e = float(log_ret[t].tail(252).std() * math.sqrt(252))
    mu_e    = float(log_ret[t].tail(252).mean() * 252)
    print(f'{t}: σ_E = {sigma_e:.3f} ({sigma_e:.1%})   μ̂ = {mu_e:.3f} ({mu_e:.1%} p.a.)')

NVDA: σ_E = 0.339 (33.9%)   μ̂ = 0.514 (51.4% p.a.)
TSLA: σ_E = 0.388 (38.8%)   μ̂ = 0.076 (7.6% p.a.)


In [14]:
# ── Merton PD and equity/debt decomposition ────────────────────────────────
merton_inputs = {
    'NVDA': {'equity_mktcap': 15.0, 'lt_debt': 1.3, 'T': 5, 'r': 0.02},
    'TSLA': {'equity_mktcap': 31.0, 'lt_debt': 2.7, 'T': 5, 'r': 0.02},
}

rows = []
for t, p in merton_inputs.items():
    E_mktcap = p['equity_mktcap']
    B        = p['lt_debt']
    T        = p['T']
    r        = p['r']
    V0       = E_mktcap + B          # simplified asset value
    leverage = E_mktcap / V0

    sigma_e  = float(log_ret[t].tail(252).std() * math.sqrt(252))
    mu_e     = float(log_ret[t].tail(252).mean() * 252)
    sigma_A  = sigma_e * leverage    # leverage-adjusted asset vol

    pd_q = merton_pd(V0, B, r,   sigma_A, T)   # Q-measure (risk-neutral)
    pd_p = merton_pd(V0, B, mu_e, sigma_A, T)  # P-measure (historical drift)
    E_M  = merton_equity(V0, B, r, sigma_A, T)
    D_M  = merton_debt(V0, B, r, sigma_A, T)

    rows.append({
        'Ticker'            : t,
        'V0 ($B)'           : V0,
        'B ($B)'            : B,
        'σ_A'               : round(sigma_A, 4),
        'μ̂ (annual)'       : round(mu_e, 4),
        'r'                 : r,
        'Q-PD (%)'          : round(pd_q * 100, 4),
        'P-PD (%)'          : round(pd_p * 100, 4),
        'Merton E₀ ($B)'    : round(E_M, 4),
        'Merton D₀ ($B)'    : round(D_M, 4),
    })

merton_df = pd.DataFrame(rows).set_index('Ticker')
print('Merton structural model results:')
display(merton_df)

# Assertions
for row in rows:
    t    = row['Ticker']
    pd_q = row['Q-PD (%)'] / 100
    E_M  = row['Merton E₀ ($B)']
    D_M  = row['Merton D₀ ($B)']
    V0   = row['V0 ($B)']
    assert 0 < pd_q < 1,                  f'{t}: Q-PD must be in (0, 1)'
    assert abs(E_M + D_M - V0) < 1e-4,   f'{t}: E₀ + D₀ must equal V₀'
    print(f'✓ {t}: Q-PD = {pd_q:.3%}  E₀ + D₀ = V₀ ✓')

Merton structural model results:


,V0 ($B),B ($B),σ_A,μ̂ (annual),r,Q-PD (%),P-PD (%),Merton E₀ ($B),Merton D₀ ($B)
Ticker,,,,,,,,,
NVDA,16.3000,1.3000,0.3119,0.5135,0.0200,0.0312,0.0000,15.1238,1.1762
TSLA,33.7000,2.7000,0.3567,0.0762,0.0200,0.1916,0.0590,31.2578,2.4422


✓ NVDA: Q-PD = 0.031%  E₀ + D₀ = V₀ ✓
✓ TSLA: Q-PD = 0.192%  E₀ + D₀ = V₀ ✓


---
## §8 — Frontend Validation Summary

This section prints all key numbers for direct comparison with the Streamlit application.

**To replicate in the frontend:**
1. **Tab 1** — enter the M7 stock positions (quantities printed above) and the three option positions.
2. **Tab 2** — upload `data/m7_2015.csv` (saved in §1).
3. **Tab 3** — set lookback=252, horizon=5, VaR conf=99%, ES conf=97.5%, estimator=window, MC paths=10 000.
4. **Tab 4** — click Run Analysis; verify VaR and ES match the table below.
5. **Tab 5** — run the historical backtest; verify Kupiec statistics match.
6. **Tab 6** — enter NVDA/TSLA Merton parameters; verify Q-PD and P-PD match.

In [15]:
sep = '=' * 76
print(sep)
print('  ADVANCED DEMO — KEY NUMBERS FOR FRONTEND VALIDATION')
print(sep)

print(f'\nEvaluation date : {EVAL_DATE}')
print(f'Portfolio       : equal-weight M7 at $50k per stock = ${V0_stocks:,.2f} (stocks)')
print(f'Full portfolio  : stocks + 3 option positions = ${V0_full:,.2f}')
print(f'Risk params     : lookback={LOOKBACK}d  horizon={HORIZON}d  VaR={VAR_CONF:.0%}  ES={ES_CONF:.1%}')

print(f'\n{"":-<76}')
print(f'{"Method":<16} {"VaR stocks":>14} {"ES stocks":>14} {"VaR full":>14} {"ES full":>14}')
print(f'{"":-<76}')
for name, rs, rf in [
    ('Historical',  r_hist,  rf_hist),
    ('Parametric',  r_param, rf_param),
    ('Monte Carlo', r_mc,    rf_mc),
]:
    vs  = rs['var'];   es_s = rs['es']
    vf  = rf['var'];   es_f = rf['es']
    print(f'{name:<16} ${vs:>12,.2f} ${es_s:>12,.2f} ${vf:>12,.2f} ${es_f:>12,.2f}')
print(f'{"":-<76}')

print(f'\nDiversification (historical VaR):')
print(f'  Sum individual VaRs : ${sum_indiv_var:,.2f}')
print(f'  Portfolio VaR       : ${port_var_hist:,.2f}')
print(f'  Benefit             : {diversif_ratio:.1%}')

print(f'\nBacktest ({kupiec["n_observations"]} obs, historical model):')
print(f'  Expected exceptions : {exp_exc:.1f}')
print(f'  Actual exceptions   : {kupiec["n_exceptions"]}')
print(f'  Kupiec LR           : {kupiec["lr_stat"]:.4f}')
print(f'  Kupiec p-value      : {kupiec["p_value"]:.4f}')
print(f'  H0 not rejected     : {not kupiec["reject_h0"]}')

print(f'\nMerton credit model:')
for row in rows:
    t = row['Ticker']
    print(f'  {t}: Q-PD = {row["Q-PD (%)"]:.4f}%   P-PD = {row["P-PD (%)"]:.4f}%')

print(sep)

  ADVANCED DEMO — KEY NUMBERS FOR FRONTEND VALIDATION

Evaluation date : 2015-12-31
Portfolio       : equal-weight M7 at $50k per stock = $349,833.69 (stocks)
Full portfolio  : stocks + 3 option positions = $351,204.44
Risk params     : lookback=252d  horizon=5d  VaR=99%  ES=97.5%

----------------------------------------------------------------------------
Method               VaR stocks      ES stocks       VaR full        ES full
----------------------------------------------------------------------------
Historical       $   25,470.58 $   27,483.82 $   26,802.60 $   29,004.40
Parametric       $   22,161.68 $   22,281.64 $   23,519.24 $   23,646.41
Monte Carlo      $   21,683.52 $   21,886.13 $   22,747.81 $   23,098.07
----------------------------------------------------------------------------

Diversification (historical VaR):
  Sum individual VaRs : $32,153.24
  Portfolio VaR       : $25,470.58
  Benefit             : 20.8%

Backtest (750 obs, historical model):
  Expected excep